# Database Hub demo: one Run all for a whole demo session

**Select Run all. There's nothing to edit.** This notebook runs a whole demo session through the demo's Azure Automation runbook (ADR 0026), and ends it, with no GitHub Actions and no script on a PC:

1. **Pre-check:** when the session ends (always before the VMs' nightly auto-shutdown), and whether the control vault answers. An earlier session whose history wasn't saved is saved right after this one's start is sent.
2. **Start:** the runbook starts the VMs (T2's guest starts with its host), T4, and T4's read replica; T3 wakes with its first query, and T5, the Azure Cosmos DB account, has no stop, so it's always there. The showcase runs on T1 to T5 for `DURATION_MINUTES` (90 by default), with the replica-lag beat on T4 and two ru-storm waves every 30 minutes on T5. T5's showcase ends after its last whole ru-storm and a few minutes of its baseline, so T5 is back at its baseline, with no throttled requests (429s), before the session ends, and the next Run all repeats its storms from the first.
3. **Follow:** each step as the runbook takes it, with the fix for anything that didn't work.
4. **Readiness:** once the session runs, `NB_DatabaseHub_Deployment`'s read-only checks run in this notebook's own Spark session and fill the Insights app's Readiness page. A check that fails or runs past 5 minutes is noted, and the session goes on.
5. **The presenter's guide:** what to open, when each beat shows in Database Hub, the Database Agent loop on T3 (the agent's issue, then this Run all's manual response), and the Copilot prompts, with this session's names and times.
6. **Live view** until the session's end, with T3's response when its time comes and T3's CPU before and after it.
7. **The end:** the runbook saves T4's, its replica's, and T5's Azure Monitor metrics, deletes the replica, stops T4, and deallocates the VMs; T5's account has no stop, so its activity ends with sqlvm01's and its throughput keeps billing. Once the telemetry has arrived, `NB_DatabaseHub_LiveActivity` from the demo workspace runs in this notebook's own Spark session and saves the session's history for the report and the Insights app. No second Spark job starts, so even an F2 or F4 capacity has nothing to refuse.

**If Run all stops early** (a closed browser, a cancelled cell), the session still ends: its own auto-stop, 10 minutes after its end, does step 7's Azure part; the next Run all saves its history first; and the nightly safety stop runs every night. A Run all while a session runs follows that session instead of starting another. **Stop now** (section 4) needs `STOP_NOW = True`, so Run all never stops a session by accident.

**What it needs:** nothing beyond Azure and Fabric, and nothing to set up by hand: `deploy.ps1` gives you Key Vault Secrets User on the control vault's three secrets and attaches the demo's lakehouse `LH_DatabaseHub` as this notebook's default lakehouse, where each history is saved. The capacity runs this notebook, so it isn't checked; you resume and pause it, and this notebook never does. The webhook's URL is a secret: it's read from the vault for one request and never printed. Run all keeps this notebook's Spark session for the whole session, about two hours for the default 90 minutes, on the capacity, and the history runs in that session too.

> **Preview disclosure:** Database Hub is generally available (2026-11-17); its Copilot, Database Agents, and alerts are previews, and so is the telemetry endpoint. Using a notebook `kusto` token against the endpoint isn't documented, so when it refuses the token the live view prints its KQL for the Azure Data Explorer web UI. SQL telemetry arrives 3 to 15 minutes after the activity, T4's and T5's Azure Monitor metrics 1 to 3 minutes after it.

**Contents:** 1 Parameters · 2 Helpers · 3 Run all: the session · 4 Stop now and Reset · 5 Links

## 1. Parameters

`deploy.ps1` filled the environment values when it imported this notebook. **For a demo, change nothing:** Run all runs a 90-minute session (`MODE = "session"`), follows it to its end (`WAIT_FOR_END = True`), saves its history, and stops everything. `MODE = "check"` shows the plan and the checks and sends nothing; `MODE = "activities"` sends only `ACTIVITIES`, a custom activity per target, as before 2026-11-14, with no replica and no history.

In [ ]:
# Parameters cell (tagged "parameters"). deploy.ps1 fills the environment values when it imports this notebook. Never
# put a credential here: the webhook URL stays in Key Vault.

# Environment (filled at import).
KEY_VAULT_URL = ""                               # https://kv-<prefix>-ctl-<suffix>.vault.azure.net/
WEBHOOK_SECRET_NAME = "dbhub-activity-webhook"   # the secrets' names, never their values
STATUS_SECRET_NAME = "dbhub-session-status"
METRICS_SECRET_NAME = "dbhub-session-metrics"
RESOURCE_GROUP = "rg-dbhub-demo"
T1_VM_NAME = "sqlvm01"
T2_MACHINE_NAME = "arcsql01"
T3_SERVER_PREFIX = "azuresql01-"
DEMO_DATABASE = "ContosoCare"
TENANT_ID = ""                                   # the demo's tenant, for the links
DEMO_WORKSPACE_ID = ""                           # the demo's Fabric workspace: the activity notebook, the report, the app
DEMO_LAKEHOUSE_ID = ""                           # its lakehouse, this notebook's default one: each history is saved there
CATALOG_JSON = ""                                # the runbook's rules: targets, scenarios, profiles, bounds, the shutdown
SHOWCASE_JSON = ""                               # the showcase's beats, for the presenter's guide

# What Run all does. "session" (the default): the whole session, from the start to the stop. "check": the plan and the
# checks only; nothing is sent. "activities": ACTIVITIES only, as before 2026-11-14 (no replica and no history; the
# run's own auto-stop ends it).
MODE = "session"
DURATION_MINUTES = 90       # session: 30-120; activities: 5-120. It always ends before the VMs' nightly auto-shutdown.
WAIT_FOR_END = True         # session: Run all follows it to its end, saves its history, and stops everything

# MODE "activities" only: one entry per target, "showcase", "baseline", "off", or a scenario from the catalog.
# "intensity" (1-10) and, for an incident, "concurrency" (1 to the scenario's maximum) are optional. A pipeline or the
# Fabric job API passes ACTIVITIES and RESET_TARGETS as JSON text, because a notebook parameter is a string, a number,
# or a bool. T5's entry is left out when T5 isn't one of the demo's targets (t5Enabled false).
ACTIVITIES = [
    {"target": "T1", "scenario": "showcase"},
    {"target": "T2", "scenario": "showcase"},
    {"target": "T3", "scenario": "showcase"},
    {"target": "T4", "scenario": "showcase"},
    {"target": "T5", "scenario": "showcase"},
]

# Section 4 needs its own switch, so Run all never stops or resets by accident. STOP_NOW ends the session now, its
# history saved first; RESET_NOW resets the targets whose VM runs (RESET_TARGETS, or all of them).
STOP_NOW = False
RESET_NOW = False
RESET_TARGETS = []

# Live view.
REFRESH_SECONDS = 60        # 30-300
LOOKBACK_MINUTES = 15       # the window each refresh shows
WATCH_MINUTES = 30          # without WAIT_FOR_END, or for MODE "activities": how long it refreshes (0: one look)

## 2. Helpers

Run the next two cells first. They change nothing:

- **Part 1, no network calls:** the request bodies the runbook accepts (`session`, `stop`, `capture`, `reset`, and `run`), checked with the runbook's own rules; the session's plan as the runbook makes it, ending before the VMs' nightly auto-shutdown and, when that cuts it short, after its last whole showcase cycle; the showcase's beats in this session's times; and the presenter's guide.
- **Part 2, the calls:** the webhook (its URL read from Key Vault for one request, never printed); the session's status and metrics, which the runbook writes to the control vault through Azure Resource Manager; `notebookutils.notebook.run`, which runs the activity notebook in this session to save a history; the Fabric REST API with your `pbi` token, for the demo workspace's items (the guide's links) and, only when the history can't run in this session, the activity notebook's job (Job Scheduler); and the telemetry endpoint with your `kusto` token, with the same KQL as the live-activity notebook (`scripts/modules/DbHubDemo/Kql`).

In [ ]:
# Helpers, part 1: requests, the session's plan, the showcase's beats, and the presenter's guide. No network calls.
import datetime as dt
import json
import re
import uuid

ACTIONS = ("run", "session", "capture", "stop", "reset")
MAX_BODY_BYTES = 4096
TELEMETRY_LAG_MINUTES = 15      # SQL telemetry reaches Database Hub 3 to 15 minutes after the activity
SQL_SHOWS_MINUTES = 3
T4_SHOWS_MINUTES = 1            # T4's Azure Monitor metrics, 1 to 3 minutes
T5_SHOWS_MINUTES = 1            # T5's Azure Monitor metrics, 1 to 3 minutes (the Hub's Cosmos DB charts: hourly points)
AGENT_ISSUE_MINUTES = 15        # Database Agents raises a new issue about 15 minutes after a condition starts (Learn)
AGENT_AFTER_MINUTES = 5         # T3's CPU after the agent beat's response is read over this many minutes, from a minute after it
AGENT_CHECK_UNTIL_MINUTES = 20  # the live view waits this long after the response for that telemetry (3 to 15 minutes late)
MIN_BEAT_MINUTES = 3            # a beat the session's end cuts shorter than this barely shows, so the guide leaves it out
ACTIVE_PHASES = ("accepted", "starting", "replica", "running")
STATUS_EVERY_SECONDS = 300      # the live view reads the session's status again at most this often (a host's failure)
_NAME = re.compile(r"^[A-Za-z0-9._@/-]{1,64}$")
_SESSION_ID = re.compile(r"^[0-9a-f]{32}$")
_REDACTIONS = (
    (re.compile(r"eyJ[A-Za-z0-9_-]{5,}\.[A-Za-z0-9_-]{5,}\.[A-Za-z0-9_-]*"), "[REDACTED:jwt]"),
    (re.compile(r"(?i)(bearer\s+)[A-Za-z0-9._~+/=-]{8,}"), r"\1[REDACTED]"),
    (re.compile(r"https://[A-Za-z0-9.-]+\.azure-automation\.net/\S*"), "[REDACTED:webhook]"),
    (re.compile(r"(?i)(webhooks\?token=)[^&\s\"']+"), r"\1[REDACTED]"),
)


def redact(value):
    """Removes token-like strings and webhook URLs from text before it's printed."""
    text = "" if value is None else str(value)
    for pattern, replacement in _REDACTIONS:
        text = pattern.sub(replacement, text)
    return text


def load_catalog(text):
    """The runbook's rules from CATALOG_JSON: targets, scenarios (maxMinutes, maxConcurrency, targets), profiles
    (showcase: maxMinutes, defaultMinutes, cycleMinutes, targets), bounds, and session (minMinutes, shutdownMarginMinutes,
    autoStopBufferMinutes, and the VMs' auto-shutdown). A catalog from before 2026-11-14 has no session rules."""
    if not text:
        raise ValueError("CATALOG_JSON is empty: run ./scripts/deploy.ps1, which imports this notebook.")
    catalog = json.loads(text)
    for key in ("targets", "scenarios", "minDurationMinutes", "maxDurationMinutes"):
        if key not in catalog:
            raise ValueError("CATALOG_JSON has no %s." % key)
    catalog.setdefault("profiles", {})
    catalog.setdefault("session", {})
    return catalog


def load_showcase(text):
    """The showcase's beats from SHOWCASE_JSON: cycleMinutes, its incidents (target, scenario, startMinute,
    durationMinutes, and optional cycles, tier, and replica, as scenarios/activity-profiles.json has them; and for the
    Database Agent beat, respondPercent, goodPercent, and agentBeat), and each scenario's title and the Database Hub
    signal it fills. Empty: the guide has no timeline."""
    showcase = json.loads(text) if text else {}
    showcase.setdefault("cycleMinutes", 0)
    showcase.setdefault("incidents", [])
    showcase.setdefault("scenarios", {})
    return showcase


def as_list(value, name):
    """A list parameter, as a Python list or as JSON text: a run through the Fabric job API or a pipeline gets each
    notebook parameter as a string, a number, or a bool."""
    if value is None:
        return []
    if isinstance(value, str):
        if not value.strip():
            return []
        try:
            value = json.loads(value)
        except ValueError:
            raise ValueError("%s must be a list, or a JSON array as text." % name) from None
    if not isinstance(value, (list, tuple)):
        raise ValueError("%s must be a list, or a JSON array as text." % name)
    return list(value)


def _whole(value, low, high, name):
    if isinstance(value, bool) or not isinstance(value, int) or not low <= value <= high:
        raise ValueError("%s must be a whole number from %d to %d." % (name, low, high))
    return value


def new_session_id():
    """A session's ID: 32 lowercase hexadecimal characters, as the runbook expects."""
    return uuid.uuid4().hex


def build_request(action, activities=None, duration_minutes=None, reset_targets=None, catalog=None, requested_by="notebook",
                  session_id=None):
    """Validates a request with the runbook's rules and returns the webhook body (a dict). No network calls."""
    if action not in ACTIONS:
        raise ValueError("action must be session, run, capture, stop, or reset.")
    if not _NAME.match(str(requested_by)):
        raise ValueError("requested_by must be 1-64 letters, digits, or . _ @ / -.")
    body = {"action": action, "requestedBy": str(requested_by)}
    if session_id is not None:
        if action not in ("session", "capture", "stop"):
            raise ValueError("Only session, capture, and stop take a session ID.")
        if not isinstance(session_id, str) or not _SESSION_ID.match(session_id):
            raise ValueError("A session ID is 32 lowercase hexadecimal characters.")
        body["sessionId"] = session_id
    elif action in ("session", "capture"):
        raise ValueError("%s needs a session ID." % action)
    if action in ("stop", "capture"):
        return body
    if action == "session":
        showcase = (catalog.get("profiles") or {}).get("showcase")
        if not showcase:
            raise ValueError("The catalog has no showcase profile: run ./scripts/deploy.ps1 again.")
        low = int((catalog.get("session") or {}).get("minMinutes") or 30)
        body["durationMinutes"] = _whole(duration_minutes, low, int(showcase["maxMinutes"]), "DURATION_MINUTES")
        return body
    targets = list(catalog["targets"])
    if action == "reset":
        chosen = as_list(reset_targets, "RESET_TARGETS")
        if len(set(chosen)) != len(chosen) or any(target not in targets for target in chosen):
            raise ValueError("RESET_TARGETS must be distinct values from: %s." % ", ".join(targets))
        if chosen:
            body["targets"] = chosen
        return body
    body["durationMinutes"] = _whole(duration_minutes, int(catalog["minDurationMinutes"]), int(catalog["maxDurationMinutes"]), "DURATION_MINUTES")
    items = as_list(activities, "ACTIVITIES")
    if not 1 <= len(items) <= len(targets):
        raise ValueError("ACTIVITIES needs 1 to %d entries, one per target." % len(targets))
    seen = set()
    cleaned = []
    for item in items:
        if not isinstance(item, dict):
            raise ValueError("Each activity is an object with a target and a scenario.")
        unknown = set(item) - {"target", "scenario", "intensity", "concurrency"}
        if unknown:
            raise ValueError("An activity can't have: %s." % ", ".join(sorted(unknown)))
        target, scenario = item.get("target"), item.get("scenario")
        if target not in targets:
            raise ValueError("Each target must be one of: %s." % ", ".join(targets))
        if target in seen:
            raise ValueError("%s has more than one activity; a target runs one at a time." % target)
        seen.add(target)
        entry = {"target": target, "scenario": scenario}
        limit = 0
        profiles = catalog.get("profiles") or {}
        if scenario == "off" or (isinstance(scenario, str) and scenario in profiles):
            if scenario != "off" and target not in profiles[scenario]["targets"]:
                raise ValueError("%s doesn't run on %s." % (scenario, target))
            if "intensity" in item or "concurrency" in item:
                raise ValueError("intensity and concurrency don't apply to %s: a profile takes its own from the catalog." % scenario)
        elif scenario != "baseline":
            definition = catalog["scenarios"].get(scenario) if isinstance(scenario, str) else None
            if definition is None:
                choices = sorted(profiles) + ["baseline", "off"]
                raise ValueError("'%s' isn't in the catalog: use %s, or one of %s." % (scenario, ", ".join(choices), ", ".join(sorted(catalog["scenarios"]))))
            if target not in definition["targets"]:
                raise ValueError("%s doesn't run on %s." % (scenario, target))
            limit = int(definition["maxConcurrency"])
        if "intensity" in item:
            entry["intensity"] = _whole(item["intensity"], 1, 10, "intensity")
        if "concurrency" in item:
            if not limit:
                raise ValueError("concurrency applies to incidents only.")
            entry["concurrency"] = _whole(item["concurrency"], 1, limit, "concurrency")
        cleaned.append(entry)
    if all(entry["scenario"] == "off" for entry in cleaned):
        raise ValueError("A run needs at least one activity that isn't off; STOP_NOW stops everything.")
    body["activities"] = cleaned
    return body


def known_activities(activities, catalog):
    """ACTIVITIES without T5's entry when T5 isn't one of the catalog's targets (t5Enabled false, 2026-11-18), so the
    default list works in a demo without T5; build_request checks every other entry."""
    items = as_list(activities, "ACTIVITIES")
    if "T5" in (catalog.get("targets") or []):
        return items
    return [item for item in items if not (isinstance(item, dict) and item.get("target") == "T5")]


def describe_plan(body, catalog):
    """MODE "activities": one line per activity, how long it runs, with the scenario's or the profile's own maximum (the
    runbook also stops it before the VMs' auto-shutdown), or that the target's activity goes off."""
    lines = []
    profiles = catalog.get("profiles") or {}
    for item in body.get("activities", []):
        if item["scenario"] == "off":
            lines.append("%s: off (sent only if its VM already runs; an off never starts a VM or T4)" % item["target"])
            continue
        if item["scenario"] == "baseline":
            cap = int(catalog["maxDurationMinutes"])
        elif item["scenario"] in profiles:
            cap = int(profiles[item["scenario"]]["maxMinutes"])
        else:
            cap = int(catalog["scenarios"][item["scenario"]]["maxMinutes"])
        lines.append("%s: %s for %d minutes" % (item["target"], item["scenario"], min(body["durationMinutes"], cap)))
    return lines


def utc_text(value):
    return value.astimezone(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def parse_utc(text):
    """An ISO 8601 UTC time from the runbook's status, or None."""
    if not isinstance(text, str) or not text.strip():
        return None
    try:
        value = dt.datetime.fromisoformat(text.strip().replace("Z", "+00:00"))
    except ValueError:
        return None
    return value.replace(tzinfo=dt.timezone.utc) if value.tzinfo is None else value.astimezone(dt.timezone.utc)


def _zone(catalog):
    """The VMs' auto-shutdown time zone (its IANA name in the catalog), or UTC when it isn't there or can't be loaded."""
    name = (catalog.get("session") or {}).get("autoShutdownIana") or "UTC"
    try:
        from zoneinfo import ZoneInfo
        return ZoneInfo(name)
    except Exception:  # an unknown name or no tz database: the times show in UTC, and zone_label says so
        return dt.timezone.utc


def zone_label(catalog):
    rules = catalog.get("session") or {}
    if _zone(catalog) is dt.timezone.utc:
        return "UTC"
    return rules.get("autoShutdownTimeZone") or rules.get("autoShutdownIana") or "UTC"


def clock(value, catalog):
    """HH:MM in the VMs' auto-shutdown time zone, the demo's own clock."""
    return value.astimezone(_zone(catalog)).strftime("%H:%M") if value else "--:--"


def next_shutdown_utc(now_utc, catalog):
    """The VMs' next nightly auto-shutdown in UTC (autoShutdownTime, HHmm, in its zone), as the runbook computes it
    (Get-DbHubNextShutdownUtc); None without one in the catalog."""
    match = re.fullmatch(r"([01]\d|2[0-3])([0-5]\d)", str((catalog.get("session") or {}).get("autoShutdownTime") or ""))
    if not match:
        return None
    zone = _zone(catalog)
    local = now_utc.astimezone(zone)
    candidate = local.replace(hour=int(match.group(1)), minute=int(match.group(2)), second=0, microsecond=0)
    if candidate <= local:
        candidate = (candidate.replace(tzinfo=None) + dt.timedelta(days=1)).replace(tzinfo=zone)
    return candidate.astimezone(dt.timezone.utc)


def session_plan(now_utc, duration_minutes, catalog):
    """The session's plan as the runbook makes it (Start-DbHubSession): the showcase for duration_minutes, at most its
    maximum, ending shutdownMarginMinutes before the VMs' next auto-shutdown; one that the shutdown cuts short ends after
    its last whole showcase cycle, so no beat is cut off; fewer than minMinutes left refuses it."""
    rules = catalog.get("session") or {}
    showcase = (catalog.get("profiles") or {}).get("showcase") or {}
    asked = int(duration_minutes)
    planned = min(asked, int(showcase.get("maxMinutes") or asked))
    shutdown = next_shutdown_utc(now_utc, catalog)
    if shutdown is not None:
        latest = shutdown - dt.timedelta(minutes=int(rules.get("shutdownMarginMinutes") or 0))
        planned = min(planned, int((latest - now_utc).total_seconds() // 60))
    minimum = int(rules.get("minMinutes") or 30)
    cycle = int(showcase.get("cycleMinutes") or 0)
    whole = False
    if minimum <= planned < asked - 1 and cycle and planned >= cycle:
        planned = planned // cycle * cycle
        whole = True
    end = now_utc + dt.timedelta(minutes=max(planned, 0))
    return {"asked": asked, "minutes": planned, "ok": planned >= minimum, "minimum": minimum, "wholeCycles": whole,
            "cycleMinutes": cycle, "startUtc": now_utc, "endUtc": end, "shutdownUtc": shutdown,
            "autoStopUtc": end + dt.timedelta(minutes=int(rules.get("autoStopBufferMinutes") or 10))}


def plan_lines(plan, catalog):
    """What the session will do, in plain words, before anything is sent: when it ends, and why, and how to change it."""
    zone = zone_label(catalog)
    fix = "set autoShutdownTime in config/demo.config.json and run ./scripts/deploy.ps1"
    if plan["shutdownUtc"] is None:
        return ["This session runs %d minutes, until %s (%s)." % (plan["minutes"], clock(plan["endUtc"], catalog), zone)]
    shutdown = clock(plan["shutdownUtc"], catalog)
    if not plan["ok"]:
        return ["Nothing is started: the VMs' nightly auto-shutdown is at %s (%s), too soon for a session of at least %d minutes." % (shutdown, zone, plan["minimum"]),
                "Fix: Run all again earlier in the day, or %s." % fix]
    if plan["minutes"] < plan["asked"] - 1:
        count = plan["minutes"] // plan["cycleMinutes"] if plan["cycleMinutes"] else 0
        cycles = " It ends after %d whole %d-minute showcase %s, so no beat is cut off." % (count, plan["cycleMinutes"], "cycle" if count == 1 else "cycles") if plan["wholeCycles"] else ""
        return ["This session runs %d of the %d minutes asked: it ends at %s (%s), before the VMs' nightly auto-shutdown at %s, when the activity on the VMs ends.%s" % (plan["minutes"], plan["asked"], clock(plan["endUtc"], catalog), zone, shutdown, cycles),
                "For a full session: Run all earlier, or %s." % fix]
    return ["This session runs %d minutes, until %s (%s). The VMs' nightly auto-shutdown is at %s, and a session always ends before it; to move it, %s." % (plan["minutes"], clock(plan["endUtc"], catalog), zone, shutdown, fix)]


def _runs_in_cycle(cycles, number):
    if cycles in (None, "", "all"):
        return True
    if cycles == "odd":
        return number % 2 == 1
    if cycles == "even":
        return number % 2 == 0
    return any(part.strip().isdigit() and int(part) == number for part in str(cycles).split(","))


def _runs_on_tier(slot_tier, tier):
    if not slot_tier:
        return True
    return slot_tier == (tier if tier in ("Burstable", "GeneralPurpose") else "Burstable")


def showcase_beats(showcase, anchors, end_utc, tier="", has_replica=False, target_ends=None):
    """The showcase's beats in UTC, as the runner plans them (ActivityProfiles.Plan): each target's cycles of cycleMinutes
    start when its profile was set (anchors: target to UTC time), a slot runs in its cycles (odd, even, or listed, from
    1), on T4's tier, and with or without T4's read replica, and every beat ends by end_utc, or by its target's own end
    when it's earlier (target_ends: T5's showcase ends at its baseline, 2026-11-18). The Database Agent beat also
    has agent, regressUtc, and respondUtc. Sorted by start."""
    cycle = int(showcase.get("cycleMinutes") or 0)
    beats = []
    if cycle <= 0 or end_utc is None:
        return beats
    info = showcase.get("scenarios") or {}
    for target in sorted(anchors):
        anchor = anchors[target]
        if anchor is None:
            continue
        own_end = (target_ends or {}).get(target)
        stop = min(end_utc, own_end) if own_end is not None else end_utc
        number = 1
        while anchor + dt.timedelta(minutes=cycle * (number - 1)) < stop:
            start_of_cycle = anchor + dt.timedelta(minutes=cycle * (number - 1))
            for slot in showcase.get("incidents") or []:
                if slot.get("target") != target or not _runs_in_cycle(slot.get("cycles"), number):
                    continue
                if not _runs_on_tier(slot.get("tier"), tier):
                    continue
                if slot.get("replica") is not None and bool(slot["replica"]) != bool(has_replica):
                    continue
                start = start_of_cycle + dt.timedelta(minutes=int(slot["startMinute"]))
                end = min(start + dt.timedelta(minutes=int(slot["durationMinutes"])), stop)
                if end - start < dt.timedelta(minutes=MIN_BEAT_MINUTES):
                    continue
                details = info.get(slot["scenario"]) or {}
                shown = details.get({"T4": "t4", "T5": "t5"}.get(target, "sql")) or {}
                beat = {"target": target, "scenario": slot["scenario"], "title": details.get("title") or slot["scenario"],
                        "surface": shown.get("surface") or "", "signal": shown.get("signal") or "", "cycle": number,
                        "startUtc": start, "endUtc": end}
                if int(slot.get("respondPercent") or 0) > 0:
                    # The Database Agent beat (2026-11-18): the run regresses at goodPercent of its window and responds at
                    # respondPercent, as the scenario times both from its own start and end.
                    span = end - start
                    beat.update(agent=bool(slot.get("agentBeat")), regressUtc=start + span * int(slot.get("goodPercent") or 35) / 100,
                                respondUtc=start + span * int(slot["respondPercent"]) / 100)
                beats.append(beat)
            number += 1
    return sorted(beats, key=lambda beat: (beat["startUtc"], beat["target"], beat["scenario"]))


def session_anchors(status):
    """When each target's showcase began: the status's dispatchUtc per target (2026-11-14), or the time of its Activity
    step for a runbook from before it."""
    anchors = {}
    for target, text in ((status or {}).get("dispatchUtc") or {}).items():
        anchors[target] = parse_utc(text)
    for step in (status or {}).get("steps") or []:
        message = str(step.get("message") or "")
        target = message.split(":", 1)[0]
        if step.get("step") == "Activity" and step.get("state") == "Pass" and re.fullmatch(r"T[1-5]", target) and target not in anchors:
            anchors[target] = parse_utc(step.get("atUtc"))
    return {target: value for target, value in anchors.items() if value is not None}


def status_beats(status, showcase):
    """This session's beats from its status: T4's tier, the replica-lag beat only when the runbook planned it, and T5's
    ru-storms only until its showcase's own end (the status's t5.endUtc, 2026-11-18)."""
    anchors = session_anchors(status)
    tier = str(((status or {}).get("t4") or {}).get("tier") or "")
    planned = bool(((status or {}).get("replicaLag") or {}).get("planned"))
    t5_end = parse_utc(((status or {}).get("t5") or {}).get("endUtc"))
    return showcase_beats(showcase, anchors, parse_utc((status or {}).get("runEndUtc")), tier, has_replica=planned,
                          target_ends={"T5": t5_end} if t5_end else None)


def session_active(status, now_utc):
    """Whether the status is of a session that still runs: starting (accepted in the last 45 minutes) or running until
    its end. A status left by a runbook job that died stops counting after that."""
    if not status or status.get("phase") not in ACTIVE_PHASES or not _SESSION_ID.match(str(status.get("sessionId") or "")):
        return False
    end = parse_utc(status.get("runEndUtc"))
    if end is not None:
        return end > now_utc
    accepted = parse_utc(status.get("acceptedUtc"))
    return accepted is not None and now_utc - accepted < dt.timedelta(minutes=45)


def step_line(step, catalog):
    """One step of the session as the runbook wrote it, with its fix."""
    at = parse_utc(step.get("atUtc"))
    line = "%s  %-10s %-7s %s" % (clock(at, catalog), step.get("step") or "", step.get("state") or "", redact(step.get("message") or ""))
    if step.get("fix"):
        line += "\n                          Fix: " + redact(step["fix"])
    return line


def agent_lines(beat, names, catalog):
    """The Database Agent loop on T3 in the demo's clock (2026-11-18): the one-time agent selection, the regression, the
    agent's issue with contextual Copilot, this Run all's manual response, and the issue's close. Text only."""
    t3 = "%s/%s" % (names.get("t3Server") or T3_SERVER_PREFIX + "<id>", names.get("t3Database") or DEMO_DATABASE)
    issue = beat["regressUtc"] + dt.timedelta(minutes=AGENT_ISSUE_MINUTES)
    lines = ["The Database Agent loop on T3 (Database Agents and Copilot are previews):",
             "  Once per deployment, if it isn't done yet: Database Hub > Manage agents > Database agent settings: Enabled on, select "
             "%s, Agent policy: Flag long-running queries, Force last known good plans, and Flag high CPU; Apply, then Save. "
             "T3 is awake now; saving needs Azure write access on T3 (the account that deployed the demo has it)." % t3,
             "  %s  T3's lookup regresses to a scanning plan; FORCE_LAST_GOOD_PLAN is held off, so automatic tuning doesn't fix it first." % clock(beat["regressUtc"], catalog),
             "  %s  From about now, the agent's issue on %s, a new one each session: open it from Database Hub's issue list; in "
             "Recommended action, select Explain why, then Analyze slow query beneath the captured query." % (clock(issue, catalog), t3),
             "  %s  Respond: this Run all forces the last good plan through Query Store, the issue's manual response, and turns "
             "FORCE_LAST_GOOD_PLAN back on." % clock(beat["respondUtc"], catalog),
             "  %s  T3's CPU falls on the dashboards; the issue closes when the agent sees the condition cleared, or close it once "
             "you've checked." % clock(beat["respondUtc"] + dt.timedelta(minutes=SQL_SHOWS_MINUTES), catalog)]
    if beat["respondUtc"] < issue + dt.timedelta(minutes=3):
        lines.append("  This session's end cuts the beat short, so the response can come before the agent's issue: a session of "
                     "60 minutes or more has the whole beat.")
    return lines


def guide_lines(status, showcase, catalog, links, beats):
    """The presenter's guide: what to open, the targets' names, each beat in the demo's clock with when it shows in
    Database Hub, the Database Agent loop on T3, and the Copilot prompts with this session's names and times. Text only."""
    names = (status or {}).get("names") or {}
    t5_account = names.get("t5Account") or ""
    zone = zone_label(catalog)
    start = parse_utc((status or {}).get("acceptedUtc"))
    end = parse_utc((status or {}).get("runEndUtc"))
    lines = ["=== The presenter's guide: session %s, %s to %s (%s) ===" % (str(status.get("sessionId") or "")[:8], clock(start, catalog), clock(end, catalog), zone),
             "Open now, each in its own tab:",
             "  Database Hub: %s" % links.get("databaseHub", ""),
             "    Overview: What needs attention?, then Recent signal changes | Estate: View Microsoft SQL, then View PostgreSQL%s | "
             "Performance: the real-time dashboards for Microsoft SQL and PostgreSQL%s | Optimization: %s, ready to migrate" % (
                 ", then View Azure Cosmos DB" if t5_account else "", ", and Azure Cosmos DB with T5's account selected" if t5_account else "",
                 names.get("t2Guest") or T2_MACHINE_NAME)]
    for key, label in (("report", "The history report"), ("dashboard", "The Real-Time Dashboard"), ("app", "The Insights app")):
        if links.get(key):
            lines.append("  %s: %s" % (label, links[key]))
    t3 = "%s/%s" % (names.get("t3Server") or T3_SERVER_PREFIX + "<id>", names.get("t3Database") or DEMO_DATABASE)
    targets = "T1 %s | T2 %s | T3 %s | T4 %s" % (names.get("sqlVm") or T1_VM_NAME, names.get("t2Guest") or T2_MACHINE_NAME, t3, names.get("t4Server") or "pgflex01-<id>")
    replica = (status.get("replica") or {})
    if replica.get("state") == "Ready" and names.get("t4Replica"):
        targets += " | T4's read replica %s" % names["t4Replica"]
    if t5_account:
        targets += " | T5 %s" % t5_account
    lines.append("The targets: " + targets)
    if beats:
        lines.append("The beats in the demo's clock (%s); Database Hub shows a SQL beat about %d to %d minutes after it starts, a T4 or T5 beat about %d to 3:" % (zone, SQL_SHOWS_MINUTES, TELEMETRY_LAG_MINUTES, min(T4_SHOWS_MINUTES, T5_SHOWS_MINUTES)))
        for beat in beats:
            shows = beat["startUtc"] + dt.timedelta(minutes=shows_minutes(beat["target"]))
            where = ("%s: %s" % (beat["surface"], beat["signal"])) if beat["surface"] else beat["signal"]
            if len(where) > 110:
                where = where[:107] + "..."
            lines.append("  %s-%s  %s  %-26s from about %s  %s" % (clock(beat["startUtc"], catalog), clock(beat["endUtc"], catalog), beat["target"], beat["title"][:26], clock(shows, catalog), where))
    note = ((status.get("replicaLag") or {}).get("note") or "")
    if note:
        lines.append("Replica lag: " + redact(note))
    agent = next((beat for beat in beats if beat.get("agent")), None)
    if agent:
        lines.extend(agent_lines(agent, names, catalog))
    t5_end = parse_utc(((status or {}).get("t5") or {}).get("endUtc"))
    if t5_account and t5_end:
        lines.append("T5: its last ru-storm ends, and T5 is back at its baseline with no throttled requests (429s), by %s, before "
                     "the session ends at %s; every Run all repeats its storms from the first." % (clock(t5_end, catalog), clock(end, catalog)))
    focus = next((beat for beat in beats if beat["target"] == "T1"), None)
    focus_name = names.get("sqlVm") or T1_VM_NAME
    focus_time = ("between %s and %s UTC" % (focus["startUtc"].strftime("%H:%M"), focus["endUtc"].strftime("%H:%M"))) if focus else "in the last 30 minutes"
    lines.append("Copilot in Database Hub (preview; Database Hub > Estate > a SQL target), with this session's names and times (UTC):")
    for title, prompt in (
            ("What changed", "What changed in my databases?"),
            ("Investigate next", "Which resource should I investigate next?"),
            ("Security first", "Which security findings should I address first?"),
            ("Estate health", "Summarize the health of my SQL estate in resource group %s. What needs attention first?" % (names.get("resourceGroup") or RESOURCE_GROUP)),
            ("Most pressure", "Which of my SQL resources is under the most pressure right now, and why?"),
            ("Investigate", "Investigate what happened on %s %s." % (focus_name, focus_time)),
            ("Root cause", "Explain the root cause in customer-friendly terms, with the evidence you used."),
            ("Next action", "What's the safest next action? Don't apply it.")):
        lines.append("  [%s] %s" % (title, prompt))
    lines.append("  T4 is PostgreSQL: Copilot's prompts above cover SQL, so show T4 on Performance's PostgreSQL real-time dashboard.")
    if t5_account:
        lines.append("  T5 is Azure Cosmos DB: show it in Estate (View: Azure Cosmos DB, %s) and on Performance: Peak Norm RU %% at 100%% "
                     "and Throttled Requests (429s) during each ru-storm, then back to the baseline's level, with no 429s." % t5_account)
    return lines


def shows_minutes(target):
    """About how many minutes after a beat starts Database Hub shows it: T4's and T5's Azure Monitor metrics arrive in 1 to
    3, SQL telemetry in 3 to 15."""
    return {"T4": T4_SHOWS_MINUTES, "T5": T5_SHOWS_MINUTES}.get(target, SQL_SHOWS_MINUTES)


def session_plan_document(status, beats):
    """SESSION_PLAN for NB_DatabaseHub_LiveActivity: the session and its beats, recorded in scenario_runs so the analysis
    correlates each beat with the telemetry. The Database Agent beat also has agent, regressUtc, and respondUtc
    (2026-11-18), for the analysis's database_agent finding. Compact JSON text."""
    def entry(beat):
        item = {"target": beat["target"], "scenario": beat["scenario"], "startUtc": utc_text(beat["startUtc"]),
                "endUtc": utc_text(beat["endUtc"])}
        if beat.get("agent"):
            item.update(agent=True, regressUtc=utc_text(beat["regressUtc"]), respondUtc=utc_text(beat["respondUtc"]))
        return item

    return json.dumps({"version": 1, "sessionId": status.get("sessionId"), "startUtc": status.get("acceptedUtc"),
                       "endUtc": status.get("runEndUtc"), "beats": [entry(beat) for beat in beats]}, separators=(",", ":"))

In [ ]:
# Helpers, part 2: the webhook, the control vault, the Fabric REST API, telemetry, and the session's steps. A token, a
# secret's value, or the webhook URL is used for one request and never printed, logged, returned, or saved.
import time

import pandas as pd
import requests

FABRIC_API = "https://api.fabric.microsoft.com/v1"
FABRIC_PORTAL = "https://app.fabric.microsoft.com"
ACTIVITY_NOTEBOOK = "NB_DatabaseHub_LiveActivity"
HISTORY_TIMEOUT_SECONDS = 1800   # the history's reference run
CAPACITY_FULL_TRIES = 3          # the history job, when the capacity answers HTTP 430, a minute apart
REPORT_NAME = "RPT_DatabaseHub_History"
DASHBOARD_NAME = "RTD_DatabaseHub_LiveView"
APP_NAME = "APP_DatabaseHub_Insights"
TELEMETRY_QUERY_URL = "https://adx.centralus.arcdataservices.com/kusto/v1/rest/query"
TELEMETRY_CONNECTION_URI = "https://adx.centralus.arcdataservices.com/kusto/"
TELEMETRY_DATABASE = "ArcSqlTelemetry"
ADX_WEB_UI = "https://dataexplorer.azure.com/"
DATABASE_HUB_URL = "https://powerbi.com/workloads/fdh/databaseHub"
WEBHOOK_HTTP_HINTS = {
    400: "HTTP 400: the webhook has expired or is disabled; ./scripts/deploy.ps1 replaces it.",
    404: "HTTP 404: the webhook, its runbook, or the Automation account wasn't found; run ./scripts/deploy.ps1.",
    500: "HTTP 500: Azure Automation couldn't take the request; try again in a minute.",
}
TELEMETRY_HTTP_HINTS = {
    400: "HTTP 400: the KQL has an error, or a table or column changed during the preview.",
    401: "HTTP 401: the endpoint didn't accept the notebook's kusto token, a path that isn't documented.",
    403: "HTTP 403: you need Reader on the subscription, and Microsoft.Sql must be registered on it.",
}


def _notebookutils():
    try:
        return notebookutils  # noqa: F821 - Fabric provides notebookutils in every notebook session.
    except NameError:
        try:
            import notebookutils as module
            return module
        except ImportError:
            return None


def send_request(body):
    """Posts one request to the webhook. Prints the HTTP status and the job ID only; returns the job IDs or None."""
    payload = json.dumps(body, separators=(",", ":"))
    if len(payload.encode("utf-8")) > MAX_BODY_BYTES:
        raise ValueError("The request is larger than %d bytes." % MAX_BODY_BYTES)
    if not KEY_VAULT_URL:
        print("Manual action required: KEY_VAULT_URL is empty. Run ./scripts/deploy.ps1, which imports this notebook.")
        return None
    utils = _notebookutils()
    if utils is None:
        print("Run this notebook in Microsoft Fabric: notebookutils isn't available here.")
        return None
    url = None
    try:
        url = utils.credentials.getSecret(KEY_VAULT_URL, WEBHOOK_SECRET_NAME)
    except Exception as error:  # the message is redacted; the URL is never in it, but a vault name could be
        print("The webhook URL couldn't be read from Key Vault (%s). You need Key Vault Secrets User on the secret, which "
              "deploy.ps1 gives you (an Owner runs ./scripts/grant-safety-stop.ps1 when it couldn't), and a private vault "
              "is reached only from the control workspace." % type(error).__name__)
        return None
    try:
        response = requests.post(url, data=payload, headers={"Content-Type": "application/json"}, timeout=60)
    except requests.RequestException as error:
        print("The webhook couldn't be reached (%s); the error text isn't printed, because it can hold the URL." % type(error).__name__)
        return None
    finally:
        url = None
    if response.status_code == 202:
        try:
            jobs = list(response.json().get("JobIds", []))
        except ValueError:
            jobs = []
        print("Accepted (HTTP 202): Automation job %s." % (", ".join(jobs) or "(no ID returned)"))
        return jobs
    print(WEBHOOK_HTTP_HINTS.get(response.status_code, "HTTP %d from the webhook." % response.status_code))
    return None


def read_secret_json(name):
    """The session's status or metrics, a JSON object the runbook writes to the control vault: (object, "") or (None,
    why). The secret's value is parsed and never printed."""
    if not KEY_VAULT_URL:
        return None, "KEY_VAULT_URL is empty: run ./scripts/deploy.ps1, which imports this notebook."
    utils = _notebookutils()
    if utils is None:
        return None, "notebookutils isn't available: run this notebook in Microsoft Fabric."
    try:
        text = utils.credentials.getSecret(KEY_VAULT_URL, name)
    except Exception as error:
        return None, ("the secret %s couldn't be read (%s): you need Key Vault Secrets User on it, which deploy.ps1 gives you, "
                      "and a private vault is reached only from the control workspace" % (name, type(error).__name__))
    try:
        value = json.loads(text or "{}")
    except ValueError:
        return None, "the secret %s isn't JSON" % name
    return (value, "") if isinstance(value, dict) else (None, "the secret %s isn't a JSON object" % name)


def get_token(audience):
    """A token for a documented notebookutils audience key: pbi (the Fabric REST API) or kusto (telemetry). Never print,
    log, or save the result."""
    if audience not in ("pbi", "kusto"):
        raise ValueError("This notebook asks only for the pbi and kusto audiences.")
    utils = _notebookutils()
    if utils is None:
        raise RuntimeError("notebookutils isn't available. Run this notebook in Microsoft Fabric.")
    return utils.credentials.getToken(audience)


def fabric_call(method, url, body=None):
    """One Fabric REST API call as you (the pbi token): (HTTP status, JSON object, Location header). Status 0 when it
    couldn't be sent. Only to api.fabric.microsoft.com."""
    if not url.startswith(FABRIC_API + "/"):
        raise ValueError("This notebook calls only the Fabric REST API.")
    try:
        token = get_token("pbi")
    except Exception as error:
        return 0, {"message": "notebookutils didn't return a pbi token: " + redact(error)}, ""
    try:
        response = requests.request(method, url, json=body, headers={"Authorization": "Bearer " + token}, timeout=60)
    except requests.RequestException as error:
        return 0, {"message": "the Fabric REST API couldn't be reached: " + redact(error)}, ""
    finally:
        token = None
    try:
        payload = response.json() if response.content else {}
    except ValueError:
        payload = {}
    return response.status_code, (payload if isinstance(payload, dict) else {}), str(response.headers.get("Location") or "")


def _fabric_error(status, payload):
    return "HTTP %d%s" % (status, (", " + redact(payload.get("errorCode") or payload.get("message") or "")) if payload else "")


def fabric_items(item_type):
    """The demo workspace's items of one type (Items - List Items): name to ID. Empty when they can't be listed."""
    found, url = {}, "%s/workspaces/%s/items?type=%s" % (FABRIC_API, DEMO_WORKSPACE_ID, item_type)
    for _ in range(20):
        status, page, _ = fabric_call("GET", url)
        if status != 200:
            break
        for item in page.get("value") or []:
            found[str(item.get("displayName") or "")] = str(item.get("id") or "")
        url = str(page.get("continuationUri") or "")
        if not url.startswith(FABRIC_API + "/"):
            break
    return found


def demo_links():
    """The guide's links: Database Hub, and the demo workspace's report, Real-Time Dashboard, and Insights app."""
    ctid = ("?ctid=" + TENANT_ID.lower()) if TENANT_ID else ""
    links = {"databaseHub": "%s/workloads/fdh/databaseHub%s" % (FABRIC_PORTAL, ctid)}
    if not DEMO_WORKSPACE_ID:
        return links
    workspace = DEMO_WORKSPACE_ID.lower()
    report = fabric_items("Report").get(REPORT_NAME)
    if report:
        links["report"] = "%s/groups/%s/reports/%s%s" % (FABRIC_PORTAL, workspace, report.lower(), ctid)
    dashboard = fabric_items("KQLDashboard").get(DASHBOARD_NAME)
    if dashboard:
        links["dashboard"] = "%s/groups/%s/kustodashboards/%s%s" % (FABRIC_PORTAL, workspace, dashboard.lower(), ctid)
    links["app"] = "%s in the demo workspace, %s/groups/%s/list%s" % (APP_NAME, FABRIC_PORTAL, workspace, ctid)
    return links


def capacity_full(status, payload):
    """Whether Fabric refused a Spark job because the capacity's Spark cores are all in use (HTTP 430,
    TooManyRequestsForCapacity). Notebook jobs started through the API aren't queued: they're refused."""
    return status == 430 or str((payload or {}).get("errorCode") or "") == "TooManyRequestsForCapacity"


def start_history_job(parameters):
    """Runs NB_DatabaseHub_LiveActivity as a job in the demo workspace (Job Scheduler - Run On Demand Item Job, jobType
    RunNotebook), as you, with these notebook parameters: the fallback when the history can't run in this notebook's
    session. A capacity whose Spark cores are all in use refuses it (HTTP 430), so it asks up to CAPACITY_FULL_TRIES
    times, a minute apart. Returns (job URL to follow, why it didn't start, whether the capacity was full)."""
    notebook = fabric_items("Notebook").get(ACTIVITY_NOTEBOOK)
    if not notebook:
        return "", "%s isn't in the demo workspace, or the workspace couldn't be read: run ./scripts/deploy.ps1" % ACTIVITY_NOTEBOOK, False
    typed = {}
    for key, value in parameters.items():
        kind = "bool" if isinstance(value, bool) else "int" if isinstance(value, int) else "string"
        typed[key] = {"value": str(value).lower() if isinstance(value, bool) else str(value), "type": kind}
    for attempt in range(CAPACITY_FULL_TRIES):
        status, payload, location = fabric_call("POST", "%s/workspaces/%s/items/%s/jobs/instances?jobType=RunNotebook" % (FABRIC_API, DEMO_WORKSPACE_ID, notebook),
                                                {"executionData": {"parameters": typed}})
        if status == 202 and location.startswith(FABRIC_API + "/"):
            return location, "", False
        if not capacity_full(status, payload):
            return "", "the job didn't start (%s)" % _fabric_error(status, payload), False
        if attempt + 1 < CAPACITY_FULL_TRIES:
            time.sleep(60)
    return "", ("the capacity refused the job %d times: TooManyRequestsForCapacity (HTTP 430), because all its Spark cores "
                "are in use, this notebook's session among them" % CAPACITY_FULL_TRIES), True


# This Run all's history jobs (job URL: its state when last read, "" before a read), and the sessions it ended.
HISTORY_JOBS, ENDED_SESSIONS = {}, []


def history_job_state(location):
    """One look at the job (Get Item Job Instance): (status, failure reason), or ("", "") when it can't be read. Run all
    never waits for it: on an F2 or F4, Fabric queues the job until this notebook's own Spark session ends, so a wait
    would only hold both (live, 2026-11-16: 30 minutes, then the job ran 5 minutes after the session ended)."""
    status, payload, _ = fabric_call("GET", location)
    if status != 200:
        return "", ""
    return str(payload.get("status") or ""), redact(((payload.get("failureReason") or {}).get("message")) or "")


def print_new_steps(status, shown):
    """Prints each step of the session's status that isn't in shown (a set of steps already printed), and adds it."""
    for step in status.get("steps") or []:
        key = (step.get("atUtc"), step.get("step"), step.get("message"))
        if key not in shown:
            shown.add(key)
            print(step_line(step, CATALOG))


def follow_status(session_id, phases, minutes, shown):
    """Reads the session's status every 20 seconds and prints each new step, until its phase is one of phases or
    minutes pass. shown: the steps already printed (a set). Returns the last status of this session, or None."""
    deadline = time.monotonic() + minutes * 60
    last, warned = None, False
    while True:
        status, why = read_secret_json(STATUS_SECRET_NAME)
        if status is None and not warned:
            print("The session's status can't be read: %s." % why)
            warned = True
        if status and status.get("sessionId") == session_id:
            last = status
            print_new_steps(status, shown)
            if status.get("phase") in phases:
                return status
        if time.monotonic() >= deadline:
            return last
        time.sleep(20)


def show_new_steps(session_id, shown):
    """One read of the session's status in the live view, which prints each new step: a host that stopped, or whose
    start or Run Command failed, which DbHub-Monitor's re-checks add while the session runs (2026-11-16). Returns the
    status while the session runs, otherwise None, so the session's end still follows the status it ran with."""
    status, _ = read_secret_json(STATUS_SECRET_NAME)
    if not status or status.get("sessionId") != session_id:
        return None
    print_new_steps(status, shown)
    return status if status.get("phase") == "running" else None


def history_parameters(status, metrics, beats):
    """The activity notebook's parameters for this session's history: its ID, beats, window, T4's metrics, and T5's
    (the cosmos member of the runbook's document, 2026-11-18), each as its own parameter."""
    start = parse_utc(status.get("acceptedUtc")) or parse_utc((metrics or {}).get("startUtc"))
    end = parse_utc(status.get("runEndUtc")) or parse_utc((metrics or {}).get("endUtc"))
    now = dt.datetime.now(dt.timezone.utc)
    parameters = {"SESSION_ID": status.get("sessionId") or (metrics or {}).get("sessionId") or ""}
    # The environment's names, which deploy.ps1 filled in here: the activity notebook's own defaults name the owner's
    # environment (rg-dbhub-demo), and its import fills only the workspace and the lakehouse.
    parameters.update({"RESOURCE_GROUP": RESOURCE_GROUP, "T1_VM_NAME": T1_VM_NAME, "T2_MACHINE_NAME": T2_MACHINE_NAME,
                       "T3_SERVER_PREFIX": T3_SERVER_PREFIX})
    if start and end:
        parameters["WINDOW_START_UTC"] = utc_text(start - dt.timedelta(minutes=5))
        parameters["WINDOW_END_UTC"] = utc_text(min(end + dt.timedelta(minutes=5), now))
    if beats:
        parameters["SESSION_PLAN"] = session_plan_document(status, beats)
    if metrics and metrics.get("sessionId") == parameters["SESSION_ID"] and metrics.get("series"):
        # Default separators: the runbook writes the secret compact, and Fabric's secret redaction replaces a secret's
        # exact text in this session's output with [REDACTED], so this text is never the secret's own.
        parameters["POSTGRES_METRICS"] = json.dumps({key: value for key, value in metrics.items() if key != "cosmos"})
    cosmos = (metrics or {}).get("cosmos")
    if isinstance(cosmos, dict) and metrics.get("sessionId") == parameters["SESSION_ID"] and cosmos.get("series"):
        parameters["COSMOS_METRICS"] = json.dumps(cosmos)
    return parameters


def history_here():
    """Whether this notebook's own Spark session can save a history. Its default lakehouse must be the demo's
    (DEMO_LAKEHOUSE_ID, which deploy.ps1 attaches), because Fabric blocks a reference run whose notebook names another
    lakehouse than the caller's. Returns (ready, why not)."""
    if not DEMO_WORKSPACE_ID or not DEMO_LAKEHOUSE_ID:
        return False, "this notebook isn't attached to the demo's lakehouse (DEMO_LAKEHOUSE_ID is empty; ./scripts/deploy.ps1 attaches it)"
    utils = _notebookutils()
    if utils is None:
        return False, "notebookutils isn't available here"
    try:
        attached = str(dict(utils.runtime.context or {}).get("defaultLakehouseId") or "").lower()
    except Exception:
        attached = ""
    if attached != DEMO_LAKEHOUSE_ID.lower():
        return False, "this notebook's default lakehouse isn't the demo's (%s; ./scripts/deploy.ps1 attaches it)" % ("it's another one" if attached else "none is attached")
    return True, ""


def run_history_here(parameters):
    """NB_DatabaseHub_LiveActivity in this notebook's own Spark session: a reference run (notebookutils.notebook.run) of
    the notebook in the demo workspace, by the workspace's ID. Fabric submits no second Spark job for it, so a small
    capacity (F2, F4) has nothing to refuse. Returns (done, its exit value or why it didn't finish)."""
    arguments = dict(parameters)
    arguments.update({"WORKSPACE_ID": DEMO_WORKSPACE_ID.lower(), "LAKEHOUSE_ID": DEMO_LAKEHOUSE_ID.lower()})
    try:
        value = _notebookutils().notebook.run(ACTIVITY_NOTEBOOK, HISTORY_TIMEOUT_SECONDS, arguments, DEMO_WORKSPACE_ID.lower())
    except Exception as error:
        return False, "it didn't finish in this session (%s: %s)" % (type(error).__name__, redact(error)[:300])
    return True, redact(value or "")[:200]


def save_history(parameters, wait=True):
    """Saves a session's history with NB_DatabaseHub_LiveActivity, which skips a session already saved. It runs in this
    notebook's own Spark session when it can, so the capacity admits no second Spark job; otherwise as a job in the demo
    workspace, which Run all never waits for while this session holds the capacity: with wait, one look says how it
    stands (history_job_state). Returns the history's state."""
    session = str(parameters.get("SESSION_ID", ""))
    ready, why = history_here()
    if ready:
        print("%s runs in this notebook's Spark session to save session %s's history." % (ACTIVITY_NOTEBOOK, session[:8]))
        done, note = run_history_here(parameters)
        if done:
            print("Pass: the history is saved%s; the report and the Insights app show it after their next refresh." % ((" (%s)" % note) if note else ""))
            return "Completed"
        why = note
    print("%s runs as a job in the demo workspace instead, because %s." % (ACTIVITY_NOTEBOOK, why))
    location, failure, full = start_history_job(parameters)
    if not location:
        if full:
            fix = ("the next Run all saves it first, while its metrics are in the control vault (until the next session "
                   "ends). Or, once another Spark session has ended (the Monitoring hub lists them), run %s in the demo "
                   "workspace with SESSION_ID = \"%s\", which saves it without T4's and T5's metrics." % (ACTIVITY_NOTEBOOK, session))
        else:
            fix = "run %s in the demo workspace with SESSION_ID = \"%s\", or Run all here again, which saves it first." % (ACTIVITY_NOTEBOOK, session)
        print("Warning: the session's history wasn't saved: %s. Fix: %s" % (failure, fix))
        return "NotStarted"
    HISTORY_JOBS[location] = ""
    print("%s runs as a job in the demo workspace to save session %s's history." % (ACTIVITY_NOTEBOOK, session[:8]))
    if not wait:
        return "Started"
    state, reason = history_job_state(location)
    HISTORY_JOBS[location] = state
    if state == "Completed":
        print("Pass: the history is saved; the report and the Insights app show it after their next refresh.")
        return state
    if state == "Deduped":
        print("Warning: %s was already running, so this run was skipped; that run saves what it finds." % ACTIVITY_NOTEBOOK)
        return state
    if state in ("Failed", "Cancelled"):
        print("Warning: the history job is %s%s. Fix: open its run in the demo workspace's Monitor hub; the next Run all "
              "saves the session's history again." % (state, (": " + reason) if reason else ""))
        return state
    where = ("Its result: %s's run in the demo workspace's Monitor hub; the report and the Insights app show the session after "
             "their next refresh. If it fails, the next Run all saves this session's history again." % ACTIVITY_NOTEBOOK)
    if state == "NotStarted":
        print("The history job is queued: Fabric starts it once this notebook's Spark session ends (a small capacity, such as "
              "an F2 or F4, runs one Spark session at a time), and Run all ends the session when it finishes. " + where)
    else:
        print("The history job is %s. %s" % (state or "started, but its state couldn't be read", where))
    return "Started"


def end_session_for_history():
    """Run all's last step, after a session's end here (finish_session). On a small capacity (F2, F4), Fabric starts a
    history job only once this notebook's own Spark session ends (live, 2026-11-16), so an interactive Run all ends the
    session now (notebookutils.session.stop, after which no code runs); a scheduled or pipeline run's session ends by
    itself. One look at a job not read yet. Returns whether it ended the session."""
    if not ENDED_SESSIONS or not HISTORY_JOBS:
        return False
    for location, state in list(HISTORY_JOBS.items()):
        if not state:
            HISTORY_JOBS[location] = history_job_state(location)[0]
    if not any(state in ("", "NotStarted") for state in HISTORY_JOBS.values()):
        return False
    utils = _notebookutils()
    try:
        interactive = bool(dict(utils.runtime.context or {}).get("isForInteractive"))
    except Exception:
        interactive = False
    if not interactive:
        return False
    print("This notebook's Spark session ends now, so Fabric can start the queued history job; the outputs above stay, and "
          "Run all again starts a new session.")
    utils.session.stop()
    return True


def finish_session(session_id, beats=None, why="the session's end"):
    """Ends a session: Stop now through the runbook, which saves T4's, its replica's, and T5's metrics first, deletes the
    replica, stops T4, and deallocates the VMs (T5 has no stop); then, once the telemetry has arrived, saves the
    session's history in this notebook's Spark session."""
    print("=== %s: stopping session %s ===" % (why[:1].upper() + why[1:], session_id[:8]))
    if not send_request(build_request("stop", session_id=session_id)):
        print("Fix: run ./scripts/stop-demo.ps1 from a PC, or wait for the session's auto-stop; the next Run all saves its history.")
        return
    stopped = follow_status(session_id, ("stopped",), 25, set())
    if stopped and stopped.get("phase") == "stopped":
        print("Stopped: %s." % ", ".join(redact(item) for item in stopped.get("done") or []))
        for item in stopped.get("failed") or []:
            print("Warning: %s. Fix: ./scripts/status.ps1 shows what still runs; ./scripts/stop-demo.ps1 stops it." % redact(item))
        metrics_note = (stopped.get("metrics") or {}).get("message") or ""
        if metrics_note:
            print("The history's metrics: " + redact(metrics_note))
    else:
        print("Warning: the stop's result didn't arrive in 25 minutes. Fix: ./scripts/status.ps1 shows what still runs.")
    status = dict(LAST_STATUS or {})
    status.setdefault("sessionId", session_id)
    metrics, _ = read_secret_json(METRICS_SECRET_NAME)
    end = parse_utc(status.get("runEndUtc")) or dt.datetime.now(dt.timezone.utc)
    ready = min(end, dt.datetime.now(dt.timezone.utc)) + dt.timedelta(minutes=TELEMETRY_LAG_MINUTES)
    wait = (ready - dt.datetime.now(dt.timezone.utc)).total_seconds()
    if wait > 0:
        print("Waiting %d minutes for the last telemetry to arrive (until %s); the VMs and T4 are already stopped." % (int(wait // 60) + 1, clock(ready, CATALOG)))
        time.sleep(wait)
    ENDED_SESSIONS.append(session_id)
    save_history(history_parameters(status, metrics, beats or []))


def kql_string(value):
    return "'" + str(value).replace("\\", "\\\\").replace("'", "\\'") + "'"


def kql_datetime(value):
    return "datetime(" + value.astimezone(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ") + ")"


def kql_prelude(start, end, bin_minutes=1):
    """Time window, bin, names, and a function that maps telemetry rows to T1, T2, or T3 (Kql/Live.Prelude.kql)."""
    return "\n".join([
        "let _start = %s;" % kql_datetime(start),
        "let _end = %s;" % kql_datetime(end),
        "let _bin = %dm;" % int(bin_minutes),
        "let _rg = %s;" % kql_string(RESOURCE_GROUP),
        "let _db = %s;" % kql_string(DEMO_DATABASE),
        "let _t1 = %s;" % kql_string(T1_VM_NAME),
        "let _t2 = %s;" % kql_string(T2_MACHINE_NAME),
        "let _t3prefix = %s;" % kql_string(T3_SERVER_PREFIX),
        "let _t3suffix = %s;" % kql_string("/" + DEMO_DATABASE),
        "let _target = (resourceType: string, resourceName: string) {",
        "    case(resourceType == 'virtualMachines' and resourceName =~ _t1, 'T1',",
        "         resourceType == 'machines' and resourceName =~ _t2, 'T2',",
        "         resourceType == 'servers/databases' and resourceName startswith _t3prefix and resourceName endswith _t3suffix, 'T3',",
        "         '')",
        "};",
    ])


def _to_utc(series):
    try:
        return pd.to_datetime(series, utc=True, format="ISO8601")
    except (TypeError, ValueError):
        return pd.to_datetime(series, utc=True, errors="coerce")


def kusto_frame(body):
    """Turns a Kusto v1 response (Tables[0].Columns and Rows) into a DataFrame."""
    tables = body.get("Tables") or []
    if not tables:
        return pd.DataFrame()
    first = tables[0]
    columns = first.get("Columns", [])
    frame = pd.DataFrame(first.get("Rows", []), columns=[column.get("ColumnName") for column in columns])
    for column in columns:
        if str(column.get("ColumnType") or column.get("DataType")).lower() in ("datetime", "system.datetime"):
            frame[column["ColumnName"]] = _to_utc(frame[column["ColumnName"]])
    return frame


def post_kql(kql):
    """Sends read-only KQL to the telemetry endpoint. Returns (status, frame, message)."""
    try:
        token = get_token("kusto")
    except Exception as error:
        return 0, None, "notebookutils didn't return a kusto token: " + redact(error)
    try:
        response = requests.post(TELEMETRY_QUERY_URL, json={"db": TELEMETRY_DATABASE, "csl": str(kql)},
                                 headers={"Authorization": "Bearer " + token, "Content-Type": "application/json"}, timeout=180)
    except requests.RequestException as error:
        return 0, None, "The telemetry endpoint couldn't be reached: " + redact(error)
    finally:
        token = None
    if response.status_code == 200:
        try:
            return 200, kusto_frame(response.json()), ""
        except ValueError:
            return 0, None, "The telemetry endpoint returned something other than Kusto v1 JSON."
    return response.status_code, None, TELEMETRY_HTTP_HINTS.get(response.status_code, "HTTP %d." % response.status_code)


# The same KQL as scripts/modules/DbHubDemo/Kql/Live.*.kql, without comments; tests/unit/Fabric.ControlNotebook.Tests.ps1
# fails if they drift.
CPU_SUMMARY_BODY = """
SqlServerCPUUtilization
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| extend CpuPercent = iff(ResourceType =~ 'servers/databases', AvgCPUPercent, coalesce(todouble(SQLProcessCPUPercent), AvgCPUPercent))
| summarize AvgCPU = round(avg(CpuPercent), 1), P95CPU = round(percentile(CpuPercent, 95), 1),
            MaxCPU = round(max(CpuPercent), 1), PctTimeAbove80 = round(100.0 * countif(CpuPercent > 80.0) / count(), 1)
        by Target, ResourceType, ResourceName, SqlServerInstanceName
| order by Target asc"""
WAITS_BODY = """
SqlServerWaitStats
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| where WaitCategory !in ('Idle', 'Unknown')
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| summarize (FirstT, FirstW, FirstC) = arg_min(SampleTimeUTC, WaitTimeMs, WaitingTasksCount),
            (LastT, LastW, LastC) = arg_max(SampleTimeUTC, WaitTimeMs, WaitingTasksCount)
        by Target, ResourceID, SqlServerInstanceName, WaitCategory, WaitType, Bucket = bin(SampleTimeUTC, _bin)
| extend T = pack_array(FirstT, LastT), W = pack_array(FirstW, LastW), C = pack_array(FirstC, LastC)
| mv-expand T to typeof(datetime), W to typeof(long), C to typeof(long)
| distinct Target, ResourceID, SqlServerInstanceName, WaitCategory, WaitType, Bucket, T, W, C
| sort by ResourceID asc, SqlServerInstanceName asc, WaitType asc, T asc
| extend Keep = ResourceID == prev(ResourceID) and SqlServerInstanceName == prev(SqlServerInstanceName)
                and WaitType == prev(WaitType) and W >= prev(W) and C >= prev(C)
| extend WaitMs = iff(Keep, W - prev(W), long(0)), Tasks = iff(Keep, C - prev(C), long(0))
| summarize WaitMs = sum(WaitMs), Tasks = sum(Tasks) by Target, WaitCategory, PageIoLatch = WaitType startswith 'PAGEIOLATCH', Bucket
| where WaitMs > 0
| order by Bucket asc
| take 50000"""
GAUGES_BODY = """
SqlServerPerformanceCountersCommon
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| where CounterType == 65792
| where CounterName in ('Processes blocked', 'Active Transactions', 'User Connections')
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| summarize MaxValue = max(CounterValue), AvgValue = avg(CounterValue)
        by Target, ResourceName, SqlServerInstanceName, CounterName, Bucket = bin(SampleTimeUTC, _bin)
| order by Bucket asc
| take 50000"""


def live_snapshot(lookback_minutes):
    """The last lookback_minutes per target: CPU, the top wait categories, and blocking. Returns (frames, message)."""
    end = dt.datetime.now(dt.timezone.utc)
    start = end - dt.timedelta(minutes=int(lookback_minutes))
    prelude = kql_prelude(start, end, 1)
    frames = {}
    for key, body in (("cpu", CPU_SUMMARY_BODY), ("waits", WAITS_BODY), ("gauges", GAUGES_BODY)):
        status, frame, message = post_kql(prelude + body)
        if status != 200:
            return None, message + " Paste this KQL into the Azure Data Explorer web UI (%s, connection %s, database %s):\n%s" % (ADX_WEB_UI, TELEMETRY_CONNECTION_URI, TELEMETRY_DATABASE, prelude + body)
        frames[key] = frame
    return frames, ""


def t3_cpu(start, end):
    """T3's average CPU percent between start and end, from the live view's CPU query. Returns (percent, or None when T3
    has no telemetry in the window or it couldn't be read; why it couldn't be read, or "")."""
    status, frame, message = post_kql(kql_prelude(start, end, 1) + CPU_SUMMARY_BODY)
    if status != 200:
        return None, message
    if frame is None or frame.empty or "Target" not in frame.columns:
        return None, ""
    rows = frame[frame["Target"] == "T3"]
    if rows.empty:
        return None, ""
    return float(rows["AvgCPU"].max()), ""


def agent_view(beat, now, state, telemetry=True):
    """The Database Agent loop in the live view (2026-11-18): a line when T3's response runs, then, once T3's telemetry from
    after it has arrived, its CPU before and after the response, with Pass or a Warning. state: the dict it keeps, whose
    "result" the session's summary prints. Read-only."""
    respond = beat["respondUtc"]
    if now < respond or state.get("result"):
        return
    if not state.get("announced"):
        state["announced"] = True
        print("Now: T3 Respond, the Database Agent loop's manual response: T3's run forces the last good plan through Query "
              "Store and turns FORCE_LAST_GOOD_PLAN back on. T3's CPU falls on the dashboards from about %s; the agent's issue "
              "then closes, or close it once you've checked." % clock(respond + dt.timedelta(minutes=SQL_SHOWS_MINUTES), CATALOG))
    if not telemetry:
        state["result"] = ("Note", "T3's CPU around the response isn't checked here, because the telemetry endpoint didn't answer "
                           "this notebook (above); Database Hub's Performance page shows it.")
        print("%s: %s" % state["result"])
        return
    after_end = respond + dt.timedelta(minutes=1 + AGENT_AFTER_MINUTES)
    if now < after_end + dt.timedelta(minutes=SQL_SHOWS_MINUTES):
        return
    after, why = t3_cpu(respond + dt.timedelta(minutes=1), after_end)
    if after is None and not why and now < respond + dt.timedelta(minutes=AGENT_CHECK_UNTIL_MINUTES):
        return
    before, why_before = t3_cpu(respond - dt.timedelta(minutes=8), respond - dt.timedelta(minutes=1)) if after is not None else (None, "")
    why = why or why_before
    if after is None or before is None:
        state["result"] = ("Warning", "T3's CPU around the response at %s isn't in the telemetry%s, so its drop isn't checked "
                           "here. Fix: Database Hub's Performance page and the Real-Time Dashboard show it once the telemetry "
                           "arrives." % (clock(respond, CATALOG), (": " + why) if why else " yet"))
    elif after <= before * 0.75 or before - after >= 10:
        state["result"] = ("Pass", "T3's CPU fell from %.0f%% before the response to %.0f%% after it: the last good plan is "
                           "back." % (before, after))
    else:
        state["result"] = ("Warning", "T3's CPU didn't fall after the response (%.0f%% before, %.0f%% after), so the regression "
                           "didn't load T3 enough to show the drop. The agent's issue and its analysis still show in Database "
                           "Hub. Fix: ./scripts/status.ps1 shows T3 and FORCE_LAST_GOOD_PLAN; the next session repeats the "
                           "beat." % (before, after))
    print("%s: %s" % state["result"])


def agent_summary(beat, state, now):
    """The Database Agent loop's line in the session's summary (2026-11-18), or none without the beat."""
    if not beat:
        return []
    if now < beat["respondUtc"]:
        return ["The Database Agent loop: the session ended before T3's response at %s, so automatic tuning wasn't held "
                "off any more and the response didn't run; the next session repeats the beat." % clock(beat["respondUtc"], CATALOG)]
    result = state.get("result")
    checked = ("%s: %s" % result) if result else "T3's CPU after it wasn't checked here; Database Hub's Performance page shows it."
    return ["The Database Agent loop: T3's response ran at %s (the last good plan forced through Query Store, "
            "FORCE_LAST_GOOD_PLAN back on). %s" % (clock(beat["respondUtc"], CATALOG), checked)]


def show_snapshot(frames):
    """Prints one compact view per refresh: CPU, the top wait categories, and blocked processes per target."""
    cpu, waits, gauges = frames["cpu"], frames["waits"], frames["gauges"]
    if not cpu.empty:
        print(cpu[["Target", "AvgCPU", "MaxCPU", "PctTimeAbove80"]].to_string(index=False))
    if not waits.empty:
        top = (waits.groupby(["Target", "WaitCategory"], as_index=False)["WaitMs"].sum()
               .assign(WaitSeconds=lambda f: (f["WaitMs"] / 1000.0).round(1))
               .sort_values(["Target", "WaitSeconds"], ascending=[True, False]).groupby("Target").head(5))
        print(top[["Target", "WaitCategory", "WaitSeconds"]].to_string(index=False))
    if not gauges.empty:
        blocked = gauges[gauges["CounterName"] == "Processes blocked"].groupby("Target", as_index=False)["MaxValue"].max()
        print(blocked.rename(columns={"MaxValue": "MaxBlockedProcesses"}).to_string(index=False))
    if cpu.empty and waits.empty and gauges.empty:
        print("No telemetry in the window yet: it arrives 3 to 15 minutes after the activity.")

## 3. Run all: the session

**Resource-changing.** With the defaults, the seven cells below run the whole session, and you only watch:

1. **Pre-check** (read-only): the plan, with when the session ends and why; a session that already runs, which this Run all then follows instead of starting another; and the control vault. The capacity isn't checked: it runs this notebook. An earlier session whose history is still in the vault is saved right after the start is sent, in this notebook's Spark session (a session already saved is skipped).
2. **Start:** one request to the runbook, which refuses it while another run is active.
3. **Follow:** the runbook's steps as it takes them, with the fix for anything that didn't work: the VMs, T4, the read replica, each target's activity, the replica-lag beat, and when T5's showcase ends at its baseline.
4. **Readiness:** once the session runs, `NB_DatabaseHub_Deployment`'s read-only checks, in this notebook's own Spark session (`SAVE_RESULTS` on, its cleanup off), which fill the Insights app's Readiness page; a validator that fails or runs past `VALIDATOR_TIMEOUT_SECONDS` is noted, and the session goes on.
5. **The presenter's guide:** printed once the session runs, with the Database Agent loop on T3: the one-time agent selection, when the agent's issue shows, and when this Run all responds.
6. **Live view:** each target's CPU, waits, and blocking every `REFRESH_SECONDS`, and each beat as it begins, until the session's end. At T3's response it says so, and once the telemetry has arrived it compares T3's CPU before and after it (Pass, or a Warning with why); the session's summary repeats that line.
7. **The end:** Stop now through the runbook (T4's and T5's metrics saved, the replica deleted, T4 stopped, the VMs deallocated), then the history, once the telemetry has arrived, in this notebook's own Spark session. When that can't run there, it runs as a job in the demo workspace, and a capacity whose Spark cores are all in use can refuse that job (HTTP 430); the line then says so, with the fix.

With `WAIT_FOR_END = False`, Run all ends after the guide and `WATCH_MINUTES` of live view; the session's auto-stop ends it, and the next Run all saves its history first.

In [ ]:
# Pre-check: the plan, a session that already runs, the control vault, and an earlier session's history. It starts
# nothing. The capacity isn't checked: it runs this notebook, and only you resume or pause it.
CATALOG = load_catalog(CATALOG_JSON)
SHOWCASE = load_showcase(SHOWCASE_JSON)
NOW = dt.datetime.now(dt.timezone.utc)
PLAN, REQUEST, JOBS, SESSION_ID, LAST_STATUS, BEATS, SHOWN, LEFTOVER = None, None, None, "", None, [], set(), None
FOLLOWING = False   # True once this Run all follows a session: one it started, or one that already ran
AGENT_BEAT, AGENT_LOOP = None, {}   # the Database Agent beat on T3 (2026-11-18), and what the live view saw of it
GO = MODE in ("session", "check", "activities") and not (STOP_NOW or RESET_NOW)
if MODE not in ("session", "check", "activities"):
    print('MODE must be "session", "check", or "activities", so nothing was sent.')
elif STOP_NOW or RESET_NOW:
    print("STOP_NOW or RESET_NOW is set, so Run all starts nothing: section 4 sends that request.")
elif MODE == "activities":
    REQUEST = build_request("run", known_activities(ACTIVITIES, CATALOG), DURATION_MINUTES, catalog=CATALOG)
    for line in describe_plan(REQUEST, CATALOG):
        print(line)
else:
    status, why = read_secret_json(STATUS_SECRET_NAME)
    if status is None:
        print("Manual action required: the control vault didn't answer: %s. Fix: ./scripts/deploy.ps1 sets the control up "
              "and gives you access; in the control workspace, the managed private endpoint to the vault must be Approved." % why)
        GO = False
    elif session_active(status, NOW):
        SESSION_ID, LAST_STATUS, FOLLOWING = status["sessionId"], status, True
        print("A session already runs (session %s, until %s, %s): this Run all follows it and ends it at its end, rather "
              "than starting another." % (SESSION_ID[:8], clock(parse_utc(status.get("runEndUtc")), CATALOG) if status.get("runEndUtc") else "starting", zone_label(CATALOG)))
    else:
        PLAN = session_plan(NOW, DURATION_MINUTES, CATALOG)
        for line in plan_lines(PLAN, CATALOG):
            print(line)
        GO = PLAN["ok"]
    if GO and MODE == "session":
        # Read now, before the start: the session that starts writes its own metrics over these when it ends.
        metrics, _ = read_secret_json(METRICS_SECRET_NAME)
        earlier = str((metrics or {}).get("sessionId") or "")
        ended = parse_utc((metrics or {}).get("endUtc"))
        # T4's series, or T5's alone (the cosmos member, 2026-11-18).
        cosmos = (metrics or {}).get("cosmos")
        kept = (metrics or {}).get("series") or (cosmos.get("series") if isinstance(cosmos, dict) else None)
        if _SESSION_ID.match(earlier) and earlier != SESSION_ID and kept and ended and NOW - ended < dt.timedelta(days=7):
            LEFTOVER = history_parameters({"sessionId": earlier}, metrics, [])
            print("An earlier session (%s, %s) left its metrics in the vault: its history is saved %s." % (
                earlier[:8], clock(ended, CATALOG), "before this Run all follows the session that runs" if FOLLOWING else "right after this session's start is sent, while the VMs start"))

In [ ]:
# Resource-changing: starts the session (MODE "session"), or sends ACTIVITIES (MODE "activities"). Once per Run all.
# Then an earlier session's history, when the pre-check found one, is saved in this notebook's Spark session.
if GO and MODE == "session" and not FOLLOWING:
    SESSION_ID = new_session_id()
    print("Session %s: asking the runbook to start it." % SESSION_ID[:8])
    if send_request(build_request("session", duration_minutes=DURATION_MINUTES, catalog=CATALOG, session_id=SESSION_ID)):
        FOLLOWING = True
    else:
        print("Nothing was started. Fix: the line above says why; ./scripts/deploy.ps1 sets the control up again.")
elif GO and MODE == "activities":
    JOBS = send_request(REQUEST)
elif GO and MODE == "check":
    print('MODE is "check", so nothing was sent.')
if LEFTOVER:
    save_history(LEFTOVER, wait=False)

In [ ]:
# Follow the session: each step as the runbook takes it, until the session runs (at most 30 minutes). Read-only.
if FOLLOWING:
    LAST_STATUS = follow_status(SESSION_ID, ("running", "refused", "failed", "stopped"), 30, SHOWN) or LAST_STATUS
    phase = (LAST_STATUS or {}).get("phase") or ""
    if phase == "running":
        print("Pass: the session runs until %s (%s); its auto-stop at %s saves T4's and T5's metrics, deletes the replica, and stops "
              "everything if this notebook can't." % (clock(parse_utc(LAST_STATUS.get("runEndUtc")), CATALOG), zone_label(CATALOG), clock(parse_utc(LAST_STATUS.get("autoStopUtc")), CATALOG)))
    elif phase in ("refused", "failed"):
        print("Nothing runs: %s" % redact(LAST_STATUS.get("message") or phase))
        FOLLOWING = False
    elif phase == "stopped":
        print("The session has stopped already (%s)." % redact(LAST_STATUS.get("stoppedBy") or "stopped"))
        FOLLOWING = False
    else:
        print("Warning: the session's status didn't say it runs within 30 minutes (last phase: %s). Fix: the runbook's "
              "output is in the control's Automation account, Jobs; when its status never arrives, the control identity "
              "can't write it yet, and an Owner runs ./scripts/grant-safety-stop.ps1 once." % (phase or "none"))
elif GO and MODE == "activities" and JOBS:
    print("A run has no status to follow: its output is in the control's Automation account, Jobs.")

In [ ]:
# Readiness, once the session runs (2026-11-18): NB_DatabaseHub_Deployment's checks in this notebook's own Spark
# session, so every Run all fills the Insights app's Readiness page (validation_results). Read-only: the validator changes
# no Azure or Fabric resource, SAVE_RESULTS only appends its results to the lakehouse, and CONFIRM and PURGE_DEMO_TABLES
# stay False, so its cleanup can't run. A validator that fails or runs past VALIDATOR_TIMEOUT_SECONDS is noted, and the
# session goes on.
VALIDATOR_NOTEBOOK = "NB_DatabaseHub_Deployment"
VALIDATOR_TIMEOUT_SECONDS = 300


def validator_parameters():
    """The validator's parameters: the demo's workspace, lakehouse, and names; its results saved; its cleanup off."""
    return {"WORKSPACE_ID": DEMO_WORKSPACE_ID.lower(), "LAKEHOUSE_ID": DEMO_LAKEHOUSE_ID.lower(), "RESOURCE_GROUP": RESOURCE_GROUP,
            "T1_VM_NAME": T1_VM_NAME, "T2_MACHINE_NAME": T2_MACHINE_NAME, "T3_SERVER_PREFIX": T3_SERVER_PREFIX,
            "SAVE_RESULTS": True, "CONFIRM": False, "PURGE_DEMO_TABLES": False}


def check_readiness():
    """Runs the validator once, as a reference run like the history's, and returns the line that says how it went. It
    never raises: readiness is a note, and the session goes on whatever happens."""
    fix = ("run %s in the demo workspace while the session runs; the Insights app's Readiness page shows its checks."
           % VALIDATOR_NOTEBOOK)
    ready, why = history_here()
    if not ready:
        return "Readiness: not checked in this Run all, because %s. Fix: %s" % (why, fix)
    started = time.monotonic()
    try:
        _notebookutils().notebook.run(VALIDATOR_NOTEBOOK, VALIDATOR_TIMEOUT_SECONDS, validator_parameters(), DEMO_WORKSPACE_ID.lower())
    except Exception as error:
        return "Warning: readiness wasn't checked: %s didn't finish in this session (%s: %s). The session goes on. Fix: %s" % (
            VALIDATOR_NOTEBOOK, type(error).__name__, redact(error)[:300].rstrip(". "), fix)
    minutes = max(1, int(round((time.monotonic() - started) / 60.0)))
    return ("Readiness: %s ran its read-only checks in %d minute%s; the Insights app's Readiness page shows each one, worst "
            "first, after its next refresh." % (VALIDATOR_NOTEBOOK, minutes, "" if minutes == 1 else "s"))


if FOLLOWING and (LAST_STATUS or {}).get("phase") == "running":
    print(check_readiness())

In [ ]:
# The presenter's guide: what to open, each beat in the demo's clock, and the Database Agent prompts. Prints text only.
if FOLLOWING and (LAST_STATUS or {}).get("phase") == "running":
    BEATS = status_beats(LAST_STATUS, SHOWCASE)
    AGENT_BEAT = next((beat for beat in BEATS if beat.get("agent")), None)
    for line in guide_lines(LAST_STATUS, SHOWCASE, CATALOG, demo_links(), BEATS):
        print(line)

In [ ]:
# Live view: read-only telemetry queries and each beat as it begins, until the session's end (WAIT_FOR_END) or for
# WATCH_MINUTES; every 5 minutes, any new step of the session's status, such as a host that stopped.
refresh = min(max(int(REFRESH_SECONDS), 30), 300)
view_end = None
if FOLLOWING and WAIT_FOR_END and (LAST_STATUS or {}).get("phase") == "running":
    view_end = parse_utc(LAST_STATUS.get("runEndUtc"))
elif GO and MODE != "check" and (FOLLOWING or JOBS):
    view_end = dt.datetime.now(dt.timezone.utc) + dt.timedelta(minutes=max(int(WATCH_MINUTES), 0))
if view_end is not None:
    announced, telemetry = set(), True
    status_due = time.monotonic() + STATUS_EVERY_SECONDS
    while True:
        now = dt.datetime.now(dt.timezone.utc)
        for beat in BEATS:
            key = (beat["target"], beat["scenario"], beat["startUtc"])
            if beat["startUtc"] <= now and key not in announced:
                announced.add(key)
                if now - beat["startUtc"] < dt.timedelta(seconds=refresh * 2):
                    print("Now: %s %s, until %s; in Database Hub from about %s." % (beat["target"], beat["title"], clock(beat["endUtc"], CATALOG),
                          clock(beat["startUtc"] + dt.timedelta(minutes=shows_minutes(beat["target"])), CATALOG)))
        if AGENT_BEAT:
            agent_view(AGENT_BEAT, now, AGENT_LOOP, telemetry)
        if telemetry:
            frames, message = live_snapshot(LOOKBACK_MINUTES)
            print("=== %s UTC, the last %d minutes" % (now.strftime("%H:%M"), int(LOOKBACK_MINUTES)))
            if frames is None:
                print(message)
                telemetry = False
            else:
                show_snapshot(frames)
        if FOLLOWING and time.monotonic() >= status_due:
            status_due = time.monotonic() + STATUS_EVERY_SECONDS
            LAST_STATUS = show_new_steps(SESSION_ID, SHOWN) or LAST_STATUS
        left = (view_end - dt.datetime.now(dt.timezone.utc)).total_seconds()
        if left <= 0:
            break
        time.sleep(min(refresh, left))

In [ ]:
# Resource-changing: the session's end. With WAIT_FOR_END, it stops the session through the runbook (T4's and T5's
# metrics saved first, the replica deleted, T4 stopped, the VMs deallocated) and then saves its history.
if FOLLOWING and WAIT_FOR_END and (LAST_STATUS or {}).get("phase") == "running":
    for line in agent_summary(AGENT_BEAT, AGENT_LOOP, dt.datetime.now(dt.timezone.utc)):
        print(line)
    finish_session(SESSION_ID, BEATS, "the session's end")
elif FOLLOWING and (LAST_STATUS or {}).get("phase") == "running":
    print("The session goes on until %s (%s). Its auto-stop at %s saves T4's and T5's metrics, deletes the replica, and stops "
          "everything; the next Run all saves its history first. To end it sooner: STOP_NOW = True, then Run all." % (
              clock(parse_utc(LAST_STATUS.get("runEndUtc")), CATALOG), zone_label(CATALOG), clock(parse_utc(LAST_STATUS.get("autoStopUtc")), CATALOG)))
elif GO and MODE == "activities" and JOBS:
    print("The run ends by itself before the VMs' auto-shutdown, and its auto-stop 10 minutes later stops the VMs and T4.")

## 4. Stop now and Reset

**Resource-changing.** **Stop now** (`STOP_NOW = True`, then Run all, or this cell) ends the session that runs as its end would: the runbook saves T4's, its replica's, and T5's metrics, deletes the replica, stops T4, deallocates the VMs, and removes the session's auto-stop (T5's account has no stop; its activity ends with sqlvm01's); then the session's history is saved once the telemetry has arrived, about 15 minutes later. Without a session, it stops whatever runs. **Reset** (`RESET_NOW = True`) runs the runner's reset on the targets whose VM runs (`RESET_TARGETS`, or all of them). `./scripts/stop-demo.ps1` stops the demo from a PC too.

In [ ]:
# Resource-changing: Stop now and Reset, each only with its own switch. Then Run all's last step.
if STOP_NOW:
    CATALOG, SHOWCASE = load_catalog(CATALOG_JSON), load_showcase(SHOWCASE_JSON)
    status, why = read_secret_json(STATUS_SECRET_NAME)
    if session_active(status, dt.datetime.now(dt.timezone.utc)):
        LAST_STATUS = status
        finish_session(status["sessionId"], status_beats(status, SHOWCASE), "Stop now")
    else:
        print("No session runs (%s), so Stop now stops whatever runs." % (why or "its status says %s" % ((status or {}).get("phase") or "none")))
        send_request(build_request("stop"))
if RESET_NOW:
    send_request(build_request("reset", reset_targets=RESET_TARGETS, catalog=load_catalog(CATALOG_JSON)))
# Run all's last step: after a session's end here, a history job queued behind this notebook's own Spark session starts
# once the session ends, so an interactive Run all ends it now. Nothing may follow it.
end_session_for_history()

## 5. Links

- **Database Hub:** Overview, Estate, and Performance, with the same telemetry as the live view: https://powerbi.com/workloads/fdh/databaseHub
- **The history:** the report `RPT_DatabaseHub_History` and the Insights app `APP_DatabaseHub_Insights` in the demo workspace show the sessions' history; `RTD_DatabaseHub_LiveView` there refreshes on its own.
- **The runbook's output:** the Automation account `aa-<prefix>-control` in the Azure portal, **Jobs**.
- **From a PC, without this notebook:** `./scripts/start-demo.ps1` and `./scripts/stop-demo.ps1 -SaveHistory`, or `./scripts/demo-control.ps1` for one activity.